# NOTEBOOK 3: REDIS

### Desde Cassandra, consulta só dúas columnas — unha será a chave e a outra o valor.

In [1]:
from cassandra.cluster import Cluster
import redis

En esta celda defino las variables de configuración necesarias para conectarme tanto a Cassandra como a Redis.  

Especifico el nodo, keyspace y tabla de Cassandra, así como las columnas que se utilizarán como clave y valor.  

Además, configuro la información del host, puerto y contraseña del servidor Redis.

In [5]:
CASSANDRA_NODES = ['localhost']
CASSANDRA_KEYSPACE = 'pipe'
CASSANDRA_TABLE = 'track'

KEY_COLUMN = 'artists'
VALUE_COLUMN = 'album_name'

REDIS_HOST = 'localhost' 
REDIS_PORT = 6379
REDIS_PASSWORD = "changeme"

Establezco una conexión con la base de datos Cassandra y ejecuto una consulta para obtener únicamente las dos columnas seleccionadas: una utilizada como clave y otra como valor.  

Cada par clave–valor extraído se almacena en la lista data_to_load, que posteriormente será cargada en Redis.  

Si ocurre algún error de conexión o consulta, el proceso se detiene y se informa en pantalla.

In [6]:
try:
    cluster = Cluster(CASSANDRA_NODES)
    session = cluster.connect(CASSANDRA_KEYSPACE)
    print(f"Conexión establecida con el Keyspace: {CASSANDRA_KEYSPACE}")

    query = f"""
    SELECT {KEY_COLUMN}, {VALUE_COLUMN} FROM {CASSANDRA_TABLE};
    """
    rows = session.execute(query)

    data_to_load = []
    for row in rows:
        key = str(getattr(row, KEY_COLUMN))
        value = str(getattr(row, VALUE_COLUMN))
        data_to_load.append((key, value))

    print(f"Datos recuperados de Cassandra: {len(data_to_load)} pares Clave-Valor.")

except Exception as e:
    print(f"Error al conectar o consultar Cassandra: {e}")
    data_to_load = [] 

Conexión establecida con el Keyspace: pipe
Datos recuperados de Cassandra: 44190 pares Clave-Valor.


### Garda estes pares chave-valor en Redis.

En esta celda verifico si existen datos obtenidos desde Cassandra y, en caso afirmativo, establezco una conexión con Redis.  

Utilizo un pipeline, que permite agrupar múltiples operaciones de escritura para ejecutarlas de forma más eficiente.  

Cada clave se almacena en Redis con el prefijo track: seguido del valor de la columna seleccionada como clave.  

Finalmente se ejecutan las operaciones y se muestra cuántos pares clave–valor fueron insertados con éxito.

In [22]:
if not data_to_load:
    print("No hay datos.")
else:
    try:
        r = redis.Redis(
            host=REDIS_HOST, 
            port=REDIS_PORT, 
            password=REDIS_PASSWORD,
            decode_responses=True)
        r.ping() 
  
        pipe = r.pipeline()
        for key, value in data_to_load:
            redis_key = key
            pipe.set(redis_key, value)
        
        results = pipe.execute()
        print(f"{len(results)}/{len(data_to_load)} pares Clave-Valor insertados en Redis.")
        
    except Exception as e:
        print(f"Error en el proceso: {type(e).__name__}: {e}")

44190/44190 pares Clave-Valor insertados en Redis.


### Verifica inserindo e consultando algunhas claves en Redis.

Se seleccionan 10 pares del conjunto de datos cargados y se consultan sus valores en Redis.  

Podemos comprobar que la inserción de datos fue correcta y que las claves son accesibles.

In [24]:
for key, _ in data_to_load[:10]:
        redis_key = key
        stored_value = r.get(redis_key)
        print(f"{redis_key} -> {stored_value}")

Reason's Why (The Very Best) -> When You Come Back Down
Rituals -> Embers
Serenity -> Zephyr
Ten Days in the Madhouse -> 青山黛瑪
The Hardcore Archive Part 2 (1995 - 1996) -> Housetime (3 Steps Ahead Remix) - Extended Mix
What's Love Got to Do with It? -> What's Love Got to Do with It
Lucky Love -> Lucky Love
Motions Beyond -> The Pressor
Bunny Girl -> Bunny Girl
飛花 -> 不知不覺愛上你
